# Artefacts in ODYSSEY (H2)

H2 (#19). The methods come from the literature collected in `docs/method.md`, *Cleaning of ODYSSEY (H2)*;
the review of #74 chose the EURADCLIM filters that run on the composite alone (Gabella, `histo_cut` on past
composites), a threshold for impossible values, and Wagner-style frequency maps for the catalogue.

1. **Gabella on rain** (review of #74, point 5): how much convective rain the filter removes on summer
   convective days, at 2 km (ODYSSEY grid) and at 1 km (CIRRUS), and where the removed rain lies.
2. **Gabella on the January features**: how much it removes on the January 2025 days whose `lk` deficit
   came from stationary features centred on radars (H1, `docs/method.md`).
3. **Pixels above 55 and 70 dBZ** on ODYSSEY and CIRRUS (review of #74, impossible values).
4. **Catalogue:** frequency maps of reflectivity classes and `histo_cut` on annual sums, per year.
5. **Static clutter across years:** how stable the `histo_cut` mask is from one year to the next.
6. **First version on ODYSSEY 2023:** the three steps of the priority in *Choice* (> 70 dBZ, Gabella,
   `histo_cut` of the previous year), with numbers and maps before and after.

In [ ]:
from opera_ds.paths import DATA_DIR, RESULTS_DIR

import csv
import glob
from collections import defaultdict

import numpy as np
from scipy import ndimage
from wradlib.classify import filter_gabella

from opera_ds.classify import obs_classes
from opera_ds.odim_io import index_files, read_odim_composite
from opera_ds.validate import read_degraded

## Settings

- **Data:** CIRRUS over the Belgian crop at :00, :15, :30 and :45. At 2 km it is degraded by `zmean`, the H1
  stand-in for ODYSSEY; at 1 km it is the real CIRRUS.
- **Filter:** `wradlib.classify.filter_gabella` with its defaults (5 × 5 window, `tr1` = 6 dB, `n_p` = 6,
  `tr2` = 1.3), `cartesian=True`, as in EURADCLIM (Overeem et al. 2023, section 3.1). Field in dBZ,
  `undetect` and values below 0 dBZ set to 0 dBZ (`thrsnorain` = 0), NaN outside coverage.
- **Classes:** Steiner on the 2 km grid, as in H1; at 1 km each 2 km class is copied to its 2 × 2 block.
- **Rain:** Marshall–Palmer, dBZ capped at 55 (Imhoff et al. 2020), on echoes ≥ 10 dBZ (the wet threshold of
  `run_validation`).
- **Where the removed rain lies:** each removed convective pixel is put in one of three groups.
  - *small echo:* its echo region (echoes above 0 dBZ, 8-connected) has at most 11 pixels, the size below
    which the area/perimeter test removes a region (Gabella and Notarpietro 2002, section 2);
  - *edge:* otherwise, when one of its 8 neighbours has no echo;
  - *interior:* the rest.

In [ ]:
WET_DBZ = 10.0
SMALL_PX = 11
EIGHT = np.ones((3, 3), bool)

files = index_files(str(DATA_DIR / "cirrus_bel" / "**" / "*@DBZH.h5"), r"@(\d{8}T\d{4})@", "%Y%m%dT%H%M")


def rain(dbz):
    return (10 ** (np.minimum(dbz, 55.0) / 10) / 200.0) ** (1 / 1.6)


def analyse(path, km):
    """Sums of rain and pixels, flagged or not, per class, and of removed convective rain per position."""
    if km == 2:
        f = read_degraded(path, factor=2).to_field(0.0)
        cls = obs_classes(f, floor=0.0, factor=1, pixel_km=2.0)
    else:
        f = read_odim_composite(path, "DBZH").to_field(0.0)
        cls = obs_classes(f, floor=0.0, factor=2, pixel_km=1.0)
    flag = filter_gabella(f.copy(), wsize=5, cartesian=True)
    echo = np.isfinite(f) & (f > 0)
    wet = np.isfinite(f) & (f >= WET_DBZ)
    r = np.where(wet, rain(np.nan_to_num(f)), 0.0)
    out = {}
    for name, m in (("all", wet), ("convective", wet & cls["convective"]), ("stratiform", wet & cls["stratiform"])):
        out[name] = np.array([r[m].sum(), r[m & flag].sum(), m.sum(), (m & flag).sum()], float)
    lab, n = ndimage.label(echo, structure=EIGHT)
    size = np.bincount(lab.ravel())
    small = echo & (size[lab] <= SMALL_PX)
    edge = echo & ~small & (ndimage.minimum_filter(echo.astype(np.uint8), footprint=EIGHT, mode="constant", cval=1) == 0)
    rem = wet & cls["convective"] & flag
    out["position"] = np.array([r[rem & small].sum(), r[rem & edge].sum(), r[rem & ~small & ~edge].sum()])
    return out


def run(days):
    tot = {km: defaultdict(lambda: defaultdict(float)) for km in (2, 1)}
    todo = [(f"{t:%Y-%m-%d}", str(p)) for t, p in sorted(files.items()) if t.minute % 15 == 0 and f"{t:%Y-%m-%d}" in days]
    print(len(todo), "frames")
    for km in (2, 1):
        for d, path in todo:
            for k, v in analyse(path, km).items():
                for key in (d, "all days"):
                    tot[km][key][k] = tot[km][key][k] + v
    return tot


def report(tot, days):
    for km in (2, 1):
        print(f"--- {km} km")
        print(f"{'day':12s} {'class':11s} {'rain removed':>13s} {'pixels removed':>15s}")
        for d in list(days) + ["all days"]:
            for k in ("all", "convective", "stratiform"):
                s, sr, n, nr = tot[km][d][k]
                print(f"{d:12s} {k:11s} {sr / s:12.1%} {nr / n:14.1%}")
        p = tot[km]["all days"]["position"]
        print("removed convective rain: small echo {:.0%}, edge {:.0%}, interior {:.0%}".format(*(p / p.sum())))

## 1. Summer convective days

The 5 days of June–July 2025 with most Steiner convective pixels in the H1 run (rows at `alpha` = 0, which
are the same for every method).

In [ ]:
days = defaultdict(int)
for mon in ("20250601-20250701", "20250701-20250801"):
    with open(RESULTS_DIR / "h1_degraded_cirrus" / mon / "per_time.csv", encoding="utf-8") as fh:
        for r in csv.DictReader(fh):
            if r["method"] == "nearest|convective" and float(r["alpha"]) == 0.0:
                days[r["time"][:10]] += int(r["n_wet"])
summer = sorted(days, key=lambda d: -days[d])[:5]
print("days (convective pixels, H1, alpha = 0):", [(d, days[d]) for d in summer])
tot_summer = run(summer)
report(tot_summer, summer)

## 2. January days with stationary features

The 5 days that carry 70% of the January `lk` deficit on convective pixels, with stationary features centred
on or near radars (`notebooks/03-mr-cirrus-analysis.ipynb`, section 4). Here the "convective" pixels are
mostly those features, so what the filter removes is a gain, not a loss.

In [ ]:
with open(RESULTS_DIR / "cirrus_analysis" / "selected_lk_deficit_times.csv", encoding="utf-8") as fh:
    january = [r["time"][:10] for r in csv.DictReader(fh)]
print("days:", january)
tot_january = run(january)
report(tot_january, january)

## 3. Pixels above 55 and 70 dBZ

Echo pixels (covered, not `undetect`) of the Belgian crops on the project PC: ODYSSEY 2023 and CIRRUS 2025.
About 40 min in one process for CIRRUS; the counts of the run of 9/11/2026 are in `docs/method.md`,
*Impossible values: counts on ODYSSEY and CIRRUS*. Set `RUN = True` to recount.

In [ ]:
RUN = False


def count(path):
    fr = read_odim_composite(path, "DBZH")
    v = fr.values[fr.coverage & ~fr.undetect]
    return np.array([v.size, ((v >= 55) & (v <= 70)).sum(), (v > 70).sum(), (v >= 100).sum(), int((v > 70).any()), 1])


if RUN:
    for name, pat in (("ODYSSEY 2023", "odyssey_bel/2023/**/*.h5"), ("CIRRUS 2025", "cirrus_bel/2025/**/*@DBZH.h5")):
        tot = sum(count(p) for p in sorted(glob.glob(str(DATA_DIR / pat), recursive=True)))
        n, a, b, c, fa, nf = tot
        print(f"{name}: {nf} files, {n:,} echo pixels; 55-70 dBZ {a:,} ({a / n:.2e}); >70 dBZ {b:,} ({b / n:.2e}), "
              f"of which >=100 dBZ {c:,}; files with any >70: {fa} ({fa / nf:.1%})")

## 4. Catalogue: frequency maps and `histo_cut`, per year

The catalogue of the review of #74, on ODYSSEY 2013–2024 and CIRRUS 2025 (`RUN_YEARS`; 2012 is left out, it
has 576 readable frames, `docs/data.md`, *Known problems*). Maps are drawn for ODYSSEY 2023 and CIRRUS 2025
(`PLOT_YEARS`); every year is printed. One pass per product and year stores, for each pixel:

- **Frequency maps** (Wagner et al. 2012, section 3, Table 1): how often the pixel falls in each reflectivity
  class: 7–19, 19–28, 28–37, 37–46, 46–55 and ≥ 55 dBZ. The paper uses 1 dBZ instead of 7 in winter; 7 is
  kept all year here.
- **Annual sum** of the rain rate (Marshall–Palmer, capped at 55 dBZ; 15 min steps for ODYSSEY, 5 min for
  CIRRUS), and the number of valid times.
- **Times above 70 dBZ**, the discard threshold proposed in the review of #74.

Then `wradlib.classify.histo_cut` on the annual sum, with the 5% tails of EURADCLIM (Overeem et al. 2023,
section 3.3) instead of the 1% default: class 1 = persistently high (static clutter), class 2 = persistently
low (shading, blockage). Under the real-time rule the mask of one year is to be applied to the following
years, not to its own; here it is only mapped.

Results are cached in `RESULTS_DIR / "h2_artefacts" / "catalogue_<product>_<year>.npz"`; one ODYSSEY year
takes about 10 min (six years at once on the project PC, 10/11/2026), one CIRRUS year about 40 min.

In [ ]:
from wradlib.classify import histo_cut

CLASS_EDGES = [7, 19, 28, 37, 46, 55]     # Wagner et al. 2012, Table 1 (lower edges, dBZ)
HISTO_FREQ = 0.05                         # Overeem et al. 2023, section 3.3
CAT = RESULTS_DIR / "h2_artefacts"
CAT.mkdir(parents=True, exist_ok=True)
PRODUCTS = {"odyssey": ("odyssey_bel", "*.h5", 0.25), "cirrus": ("cirrus_bel", "*@DBZH.h5", 5 / 60)}
RUN_YEARS = {"odyssey": list(range(2013, 2025)), "cirrus": [2025]}   # 2012 has 576 frames (data.md, Known problems)
PLOT_YEARS = {("odyssey", 2023), ("cirrus", 2025)}


def static_clutter(c):
    """histo_cut on the annual sum of the pixels with at least one valid time (wradlib's histo_cut fails
    when the input has NaN, as for the 337 pixels never covered in 2013)."""
    ok = c["valid"] > 0
    mask = np.zeros(ok.shape, np.uint8)
    mask[ok] = histo_cut(c["annual_sum"][ok], upper_frequency=HISTO_FREQ, lower_frequency=HISTO_FREQ)
    return mask


def catalogue(product, year):
    path = CAT / f"catalogue_{product}_{year}.npz"
    if path.exists():
        return dict(np.load(path))
    folder, pattern, step_h = PRODUCTS[product]
    files = sorted(glob.glob(str(DATA_DIR / folder / str(year) / "**" / pattern), recursive=True))
    freq = acc = valid = gt70 = None
    for p in files:
        fr = read_odim_composite(p, "DBZH")
        ok = fr.coverage
        v = np.where(ok & ~fr.undetect, fr.values, -np.inf)
        if freq is None:
            freq = np.zeros((len(CLASS_EDGES),) + v.shape, np.int32)
            acc, valid, gt70 = np.zeros(v.shape), np.zeros(v.shape, np.int32), np.zeros(v.shape, np.int32)
        cls = np.digitize(v, CLASS_EDGES) - 1          # -1 below 7 dBZ
        for k in range(len(CLASS_EDGES)):
            freq[k] += cls == k
        acc += np.where(np.isfinite(v), rain(np.where(np.isfinite(v), v, 0.0)), 0.0) * step_h
        valid += ok
        gt70 += v > 70
    out = {"freq": freq, "annual_sum": acc, "valid": valid, "gt70": gt70, "n_files": np.array(len(files))}
    np.savez_compressed(path, **out)
    return out


cat = {}
for product, years in RUN_YEARS.items():
    for year in years:
        c = catalogue(product, year)
        mask = static_clutter(c)
        cat[(product, year)] = (c, mask)
        print(f"{product} {year}: {int(c['n_files'])} files; annual sum median {np.median(c['annual_sum'][c['valid'] > 0]):.0f} mm, "
              f"max {c['annual_sum'].max():.0f} mm; histo_cut static clutter {int((mask == 1).sum())} pixels, "
              f"shading {int((mask == 2).sum())} pixels; pixels above 70 dBZ at least once {int((c['gt70'] > 0).sum())}")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

for (product, year), (c, mask) in cat.items():
    if (product, year) not in PLOT_YEARS:
        continue
    fig, ax = plt.subplots(1, 4, figsize=(22, 5.5), layout="constrained")
    im = ax[0].imshow(c["annual_sum"], cmap="viridis", vmax=np.percentile(c["annual_sum"], 99.5))
    ax[0].set_title(f"{product} {year}: annual sum (mm)"); fig.colorbar(im, ax=ax[0], shrink=0.8)
    im = ax[1].imshow(c["freq"][0] / np.maximum(c["valid"], 1), cmap="magma_r")
    ax[1].set_title("frequency of 7-19 dBZ"); fig.colorbar(im, ax=ax[1], shrink=0.8)
    im = ax[2].imshow(c["freq"][-1] / np.maximum(c["valid"], 1), cmap="magma_r", norm=LogNorm(vmin=1e-5, vmax=1))
    ax[2].set_title("frequency of >= 55 dBZ"); fig.colorbar(im, ax=ax[2], shrink=0.8)
    ax[3].imshow(np.where(mask > 0, mask, np.nan), cmap="coolwarm", vmin=1, vmax=2, interpolation="nearest")
    ax[3].set_title("histo_cut: static clutter (blue), shading (red)")
    for a_ in ax:
        a_.set_xticks([]); a_.set_yticks([])
    plt.show()

## 5. Static clutter across years

Under the real-time rule the `histo_cut` mask of one year is applied to the next. This only works if the
mask is stable: for each pixel, the number of ODYSSEY years (2013–2024) in which it is flagged, and the share
of each year's mask that was already flagged the year before.

In [ ]:
years = sorted(y for p, y in cat if p == "odyssey")
flagged = np.array([cat[("odyssey", y)][1] == 1 for y in years])
n_years = flagged.sum(0)
print(f"flagged at least once {int((n_years > 0).sum()):,} px; in all {len(years)} years {int((n_years == len(years)).sum()):,}; "
      f"in 10 or more {int((n_years >= 10).sum()):,}; in 1-3 years {int(((n_years > 0) & (n_years <= 3)).sum()):,}")
for a, b in zip(years[:-1], years[1:]):
    pa, pb = flagged[years.index(a)], flagged[years.index(b)]
    print(f"{b}: {int(pb.sum()):6,} px flagged, {(pa & pb).sum() / pb.sum():.0%} already flagged in {a}")

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5), layout="constrained")
im = ax[0].imshow(np.where(n_years > 0, n_years, np.nan), cmap="viridis", vmin=1, vmax=len(years), interpolation="nearest")
ax[0].set_title(f"histo_cut static clutter: years flagged (of {len(years)})"); fig.colorbar(im, ax=ax[0], shrink=0.8)
c19 = cat[("odyssey", 2019)][0]["annual_sum"]
im = ax[1].imshow(c19, cmap="viridis", vmax=np.percentile(c19, 99.5)); ax[1].set_title("ODYSSEY 2019: annual sum (index, mm)")
fig.colorbar(im, ax=ax[1], shrink=0.8)
for a_ in ax:
    a_.set_xticks([]); a_.set_yticks([])
plt.show()

### Radar positions on the mask (review of #74)

Positions from the single-radar files of January 2025 in `data/opera_pvol` (`scripts/fetch_pvol_positions.py`),
projected on the crop grid. The bucket has no UK single-radar files, so the UK radars are missing. Distances
are in km on the 2 km grid.

In [ ]:
import os
import h5py
from pyproj import Proj

ref = read_odim_composite(sorted(glob.glob(str(DATA_DIR / "odyssey_bel" / "2023" / "06" / "*.h5")))[0])
proj = Proj(ref.where["projdef"])
x_ul, y_ul = proj(float(ref.where["UL_lon"]), float(ref.where["UL_lat"]))
radars = {}
for p in sorted(glob.glob(str(DATA_DIR / "opera_pvol" / "*" / "*.h5"))):
    with h5py.File(p, "r") as f_:
        lon, lat = (float(np.ravel(f_["where"].attrs[k])[0]) for k in ("lon", "lat"))
    x, y = proj(lon, lat)
    r, c = (y_ul - y) / float(ref.where["yscale"]), (x - x_ul) / float(ref.where["xscale"])
    if -50 <= r < n_years.shape[0] + 50 and -50 <= c < n_years.shape[1] + 50:      # within 100 km of the crop
        radars[os.path.basename(p).split("@")[0]] = (int(round(r)), int(round(c)))
print(len(radars), "radars:", " ".join(sorted(radars)))

on_grid = np.zeros(n_years.shape, bool)
for r, c in radars.values():
    if 0 <= r < on_grid.shape[0] and 0 <= c < on_grid.shape[1]:
        on_grid[r, c] = True
dist_km = ndimage.distance_transform_edt(~on_grid) * 2.0
for name, m in (("crop", np.ones(n_years.shape, bool)), ("flagged >= 1 year", n_years > 0),
                ("flagged >= 10 years", n_years >= 10), ("flagged 1-3 years", (n_years > 0) & (n_years <= 3))):
    print(f"{name:20s} {int(m.sum()):7,} px; within 20 km of a radar {np.mean(dist_km[m] <= 20):6.1%}, 40 km {np.mean(dist_km[m] <= 40):6.1%}")
lab, k = ndimage.label(n_years >= 6)
size = ndimage.sum(np.ones(n_years.shape), lab, range(1, k + 1))
discs = np.where(size >= 20)[0] + 1
centres = ndimage.center_of_mass(np.ones(n_years.shape), lab, discs)
d_c = np.array([dist_km[int(round(a)), int(round(b))] for a, b in centres])
print(f"discs flagged in >= 6 years (>= 20 px): {len(discs)}; centre within 10 km of a radar: {int((d_c <= 10).sum())}")

fig, ax = plt.subplots(1, 2, figsize=(13, 5.8), layout="constrained")
im = ax[0].imshow(np.where(n_years > 0, n_years, np.nan), cmap="viridis", vmin=1, vmax=len(years), interpolation="nearest")
fig.colorbar(im, ax=ax[0], shrink=0.8, label=f"years flagged (of {len(years)})")
im = ax[1].imshow(c19, cmap="viridis", vmax=np.percentile(c19, 99.5)); fig.colorbar(im, ax=ax[1], shrink=0.8)
for a_ in ax:
    for r, c in radars.values():
        a_.plot(c, r, "o", mfc="none", mec="red", ms=9, mew=1.5)
    a_.set_xlim(0, n_years.shape[1]); a_.set_ylim(n_years.shape[0], 0); a_.set_xticks([]); a_.set_yticks([])
ax[0].set_title("histo_cut static clutter: years flagged\nred circles: radars (2025 single-radar files)", fontsize=10)
ax[1].set_title("ODYSSEY 2019: annual sum (index, mm)\nsame radars", fontsize=10)
plt.show()

## 6. First version on ODYSSEY 2023

The priority of *Choice* (`docs/method.md`), applied to each 15 min composite of 2023 in this order:

1. **Impossible values:** > 70 dBZ (Fluck et al. 2021, the convention in *Choice*) set to no rain.
2. **Gabella:** `filter_gabella` as in section 1; flagged cells set to no rain, as in EURADCLIM (Overeem et
   al. 2023, section 3.1).
3. **Static mask:** `histo_cut` (5% tails) of **2022**, the previous year, as the real-time rule requires.
   The rain of a masked cell is replaced by the inverse-distance mean (power 2) of its 4 nearest unmasked
   cells, only when that is lower (EURADCLIM, section 3.3, which does it on hourly sums; here on each
   composite).

Rain: Marshall–Palmer, capped at 55 dBZ, echoes ≥ 7 dBZ (Imhoff et al. 2020). Classes: Steiner on the
original field at 2 km, as in section 1. Belgium: the Natural Earth mask of `data/mask_BEL_1km.npy` on the
2 km crop (cells at least half inside). Sequential, about 40 min; cached in
`RESULTS_DIR / "h2_artefacts" / "first_version_odyssey_2023.npz`.

In [ ]:
from scipy.spatial import cKDTree

FV_YEAR = 2023
FV_STEP_H = 0.25
HIST_EDGES = np.arange(-10, 131, 1.0)
FV_CLASSES = ("all", "convective", "stratiform")


def rain7(dbz):
    return np.where(dbz >= 7, rain(dbz), 0.0)


def first_version(year):
    path = CAT / f"first_version_odyssey_{year}.npz"
    if path.exists():
        return dict(np.load(path))
    mask = static_clutter(catalogue("odyssey", year - 1)) == 1
    good, bad = np.argwhere(~mask), np.argwhere(mask)
    d, i = cKDTree(good).query(bad, k=4)
    nb, w = good[i], 1.0 / d ** 2
    w /= w.sum(1, keepdims=True)
    acc = {k: np.zeros(mask.shape) for k in ("before", "after70", "afterGab", "after")}
    f55 = {k: np.zeros(mask.shape, np.int32) for k in ("before", "after")}
    hist = {k: np.zeros(len(HIST_EDGES) - 1) for k in ("before", "after")}
    removed = {c: np.zeros(4) for c in FV_CLASSES}   # rain before, removed by >70, by Gabella, by the static mask
    files = sorted(glob.glob(str(DATA_DIR / "odyssey_bel" / str(year) / "**" / "*.h5"), recursive=True))
    for p in files:
        f = read_odim_composite(p, "DBZH").to_field(0.0)
        cls = obs_classes(f, floor=0.0, factor=1, pixel_km=2.0)
        r0 = rain7(np.nan_to_num(f))
        z1 = np.where(f > 70, 0.0, f)
        r1 = rain7(np.nan_to_num(z1))
        z2 = np.where(filter_gabella(z1.copy(), wsize=5, cartesian=True), 0.0, z1)
        r2 = rain7(np.nan_to_num(z2))
        r3 = r2.copy()
        r3[bad[:, 0], bad[:, 1]] = np.minimum(r2[bad[:, 0], bad[:, 1]], (r2[nb[..., 0], nb[..., 1]] * w).sum(1))
        z3 = np.where(r3 < r2, 0.0, z2)
        for k, r in (("before", r0), ("after70", r1), ("afterGab", r2), ("after", r3)):
            acc[k] += r * FV_STEP_H
        f55["before"] += np.nan_to_num(f) >= 55
        f55["after"] += np.nan_to_num(z3) >= 55
        hist["before"] += np.histogram(f[np.isfinite(f) & (f > 0)], HIST_EDGES)[0]
        hist["after"] += np.histogram(z3[np.isfinite(z3) & (z3 > 0)], HIST_EDGES)[0]
        for c in FV_CLASSES:
            m = np.ones(mask.shape, bool) if c == "all" else cls[c]
            removed[c] += np.array([r0[m].sum(), (r0 - r1)[m].sum(), (r1 - r2)[m].sum(), (r2 - r3)[m].sum()]) * FV_STEP_H
    out = {"n_files": len(files), "mask": mask, "hist_edges": HIST_EDGES}
    out.update({f"acc_{k}": v for k, v in acc.items()})
    out.update({f"f55_{k}": v for k, v in f55.items()})
    out.update({f"hist_{k}": v for k, v in hist.items()})
    out.update({f"removed_{k}": v for k, v in removed.items()})
    np.savez_compressed(path, **out)
    return out


fv = first_version(FV_YEAR)
m1 = np.load(DATA_DIR / "mask_BEL_1km.npy")[2 * 1053:2 * 1531, 2 * 528:2 * 1027]
bel = m1.reshape(478, 2, 499, 2).mean((1, 3)) >= 0.5          # crop rows 1053-1531, cols 528-1027 (data.md)
print(f"{int(fv['n_files'])} files; static mask of {FV_YEAR - 1}: {int(fv['mask'].sum()):,} px, {int((fv['mask'] & bel).sum())} in Belgium")
print(f"{'pixels':11s} {'> 70 dBZ':>9s} {'Gabella':>9s} {'static':>9s} {'total':>9s}   (share of the rain removed)")
for c in FV_CLASSES:
    b, a, g, s = fv[f"removed_{c}"]
    print(f"{c:11s} {a / b:9.2%} {g / b:9.2%} {s / b:9.2%} {(a + g + s) / b:9.2%}")
print(f"Belgium: annual sum (index) {fv['acc_before'][bel].mean():,.0f} -> {fv['acc_after'][bel].mean():,.0f} mm, "
      f"{1 - fv['acc_after'][bel].sum() / fv['acc_before'][bel].sum():.1%} removed")
nf = int(fv["n_files"])
f0, f1 = fv["f55_before"] / nf, fv["f55_after"] / nf
print(f"pixels with >= 55 dBZ in more than 0.1% of the times: {int((f0 > 1e-3).sum()):,} -> {int((f1 > 1e-3).sum()):,}; "
      f"more than 1%: {int((f0 > 1e-2).sum())} -> {int((f1 > 1e-2).sum())}")
e = fv["hist_edges"]
for lo, hi in ((0, 20), (20, 40), (40, 55), (55, 70), (70, 131)):
    s_ = (e[:-1] >= lo) & (e[:-1] < hi)
    print(f"echoes {lo:3d}-{hi:3d} dBZ kept: {fv['hist_after'][s_].sum() / max(fv['hist_before'][s_].sum(), 1):6.1%}")

fig, ax = plt.subplots(1, 4, figsize=(24, 5.6), layout="constrained")
vmax = np.percentile(fv["acc_before"], 99)
for i, (k, t) in enumerate((("acc_before", f"ODYSSEY {FV_YEAR}: annual sum before (index, mm)"),
                            ("acc_after", f"after: > 70 dBZ, Gabella, histo_cut of {FV_YEAR - 1}"))):
    im = ax[i].imshow(fv[k], cmap="viridis", vmax=vmax); ax[i].set_title(t)
fig.colorbar(im, ax=ax[:2], shrink=0.8)
for i, (f_, t) in enumerate(((f0, "frequency of >= 55 dBZ, before"), (f1, "after"))):
    im = ax[2 + i].imshow(np.where(f_ > 0, f_, np.nan), cmap="magma_r", norm=LogNorm(vmin=1e-5, vmax=1e-1)); ax[2 + i].set_title(t)
fig.colorbar(im, ax=ax[2:], shrink=0.8)
for a_ in ax:
    a_.set_xticks([]); a_.set_yticks([])
plt.show()

## 7. Radars over time (review of #74)

Radars switch on and off and change behaviour, so a yearly map mixes periods. Two sources:

- **Which radars contributed:** `how/nodes` of every crop; a radar counts on a day when it is listed in at
  least half of the day's composites (`docs/data.md`, *Radars in the ODYSSEY composites*).
- **How they behaved:** the catalogue statistics of section 4, one month at a time, around each radar of
  section 5: echo frequency (≥ 7 dBZ) in a disc of 20 km (*near*) and in a ring at 60–100 km (*far*). Rain
  varies in both, so the near/far ratio follows the radar and its processing. Months with the radar listed
  on at least 80% of the days.

The monthly catalogue is cached in `RESULTS_DIR / "h2_artefacts" / "monthly"`; about 20 min with 8 processes
for the 140 months (10/11/2026), longer in this notebook, which runs them one at a time.

In [ ]:
import os
import re
import pandas as pd

MONTHLY = CAT / "monthly"
MONTHLY.mkdir(exist_ok=True)


def monthly_catalogue(ym):
    path = MONTHLY / f"catalogue_odyssey_{ym}.npz"
    if path.exists():
        return dict(np.load(path))
    files = sorted(glob.glob(str(DATA_DIR / "odyssey_bel" / ym.replace("-", "/") / "*.h5")))
    freq = acc = valid = gt70 = None
    for p in files:
        fr = read_odim_composite(p, "DBZH")
        v = np.where(fr.coverage & ~fr.undetect, fr.values, -np.inf)
        if freq is None:
            freq = np.zeros((len(CLASS_EDGES),) + v.shape, np.int32)
            acc, valid, gt70 = np.zeros(v.shape), np.zeros(v.shape, np.int32), np.zeros(v.shape, np.int32)
        cls = np.digitize(v, CLASS_EDGES) - 1
        for k in range(len(CLASS_EDGES)):
            freq[k] += cls == k
        acc += np.where(np.isfinite(v), rain(np.where(np.isfinite(v), v, 0.0)), 0.0) * 0.25
        valid += fr.coverage
        gt70 += v > 70
    out = {"freq": freq, "annual_sum": acc, "valid": valid, "gt70": gt70, "n_files": np.array(len(files))}
    np.savez_compressed(path, **out)
    return out


def nodes_of(path):
    with h5py.File(path, "r") as f_:
        s = f_["how"].attrs.get("nodes", b"")
    s = s.decode() if isinstance(s, bytes) else str(s)
    return [n.replace("NOD:", "").strip() for n in re.split(r"[,\s]+", s) if n.strip()]


NODES_CSV = CAT / "nodes_odyssey.csv"
if not NODES_CSV.exists():                         # about 1 min with 12 processes, longer here
    rows = []
    for p in sorted(glob.glob(str(DATA_DIR / "odyssey_bel" / "*" / "*" / "*.h5"))):
        day = os.path.basename(p).split("@")[1][:8]
        rows += [(day, n) for n in nodes_of(p)]
    nd = pd.DataFrame(rows, columns=["day", "node"])
    steps = pd.Series([os.path.basename(p).split("@")[1][:8] for p in glob.glob(str(DATA_DIR / "odyssey_bel" / "*" / "*" / "*.h5"))]).value_counts()
    nd = nd.groupby(["day", "node"]).size().rename("steps_listed").reset_index()
    nd["steps_day"] = nd.day.map(steps)
    nd.to_csv(NODES_CSV, index=False)
nd = pd.read_csv(NODES_CSV)
nd["t"] = pd.to_datetime(nd.day.astype(str)).dt.to_period("M").dt.to_timestamp()
days_on = nd[nd.steps_listed >= 0.5 * nd.steps_day].groupby(["node", "t"]).day.nunique()
days_month = nd.groupby("t").day.nunique()

months = sorted({p.parent.name + "-" + p.name for p in (DATA_DIR / "odyssey_bel").glob("*/*/") if any(p.glob("*.h5"))})
yy, xx = np.mgrid[0:n_years.shape[0], 0:n_years.shape[1]]
areas = {n: (np.hypot(yy - r, xx - c) * 2.0 <= 20, (np.hypot(yy - r, xx - c) * 2.0 >= 60) & (np.hypot(yy - r, xx - c) * 2.0 <= 100))
         for n, (r, c) in radars.items()}
rows = []
for ym in months:
    m_ = monthly_catalogue(ym)
    echo = m_["freq"].sum(0) / np.maximum(m_["valid"], 1)
    t = pd.Timestamp(ym + "-01")
    for n, (near, far) in areas.items():
        if near.any() and far.any():
            share = days_on.get((n, t), 0) / days_month.get(t, np.nan)
            rows.append((t, n, echo[near].mean(), echo[far].mean(), share))
series = pd.DataFrame(rows, columns=["t", "node", "echo_near", "echo_far", "share_on"])
series["ratio"] = series.echo_near / series.echo_far
on = series[(series.share_on >= 0.8) & (series.t >= "2013-04-01")]
med = on.groupby("t")[["ratio", "echo_near", "echo_far"]].median()
print(med.loc["2015-09-01":"2016-02-01"].round(3)); print(med.loc["2018-05-01":"2018-10-01"].round(3))

steps_ = []
for n, g in on.groupby("node"):
    lr = np.log(g.sort_values("t").ratio.values); tt = g.sort_values("t").t.values
    best = max(((np.median(lr[i:i + 6]) - np.median(lr[i - 6:i]), tt[i]) for i in range(6, len(lr) - 6)), key=lambda s: abs(s[0]), default=(0, None))
    steps_.append((n, round(float(np.exp(best[0])), 2), str(best[1])[:7]))
print(pd.DataFrame(steps_, columns=["node", "largest step (factor)", "from"]).sort_values("from").to_string(index=False))

fig, ax = plt.subplots(2, 1, figsize=(14, 8), sharex=True, layout="constrained")
for n in ["bewid", "bejab", "behel", "nldhl", "nlhrw", "frabb", "deess"]:
    g = on[on.node == n]; ax[0].plot(g.t, g.ratio, ".-", lw=0.8, ms=3, label=n)
ax[0].plot(med.index, med.ratio, "k-", lw=2.5, label="median of all radars")
ax[0].set_yscale("log"); ax[0].set_ylabel("echo frequency near / far"); ax[0].legend(ncol=4, fontsize=8)
ax[1].plot(med.index, med.echo_near, "r-", label="near (<= 20 km), median of radars")
ax[1].plot(med.index, med.echo_far, "b-", label="far (60-100 km), median of radars")
ax[1].set_ylabel("echo frequency (>= 7 dBZ)"); ax[1].legend()
for a_ in ax:
    a_.axvspan(pd.Timestamp("2015-11-01"), pd.Timestamp("2016-02-01"), color="orange", alpha=0.3)
plt.show()

## 8. Isolated echoes, blocked sectors, spokes, and the OPERA hit-count rule

- **Isolated echoes, 2023:** an echo pixel is isolated when its region (echoes above 0 dBZ, 8-connected) has
  at most 11 pixels, as in section 1. Cached in `speckle_odyssey_2023.npz` (about 1 min with 12 processes,
  longer here).
- **Blocked sectors and spokes** (Wagner et al. 2012, section 3): for each radar, the median of each 1°
  azimuth in a ring at 20–80 km, of the echo frequency over September 2018–June 2024 (the last period of
  section 7), and of the 2023 annual sum before and after the first version (section 6). A change of more
  than 10% between adjacent azimuths marks a spoke edge.
- **OPERA hit-count rule** (Scovell 2013): a pixel whose monthly echo frequency exceeds 60% (90% at first).

In [ ]:
SPECKLE = CAT / "speckle_odyssey_2023.npz"
if not SPECKLE.exists():
    small_n = echo_n = None
    files23 = sorted(glob.glob(str(DATA_DIR / "odyssey_bel" / "2023" / "**" / "*.h5"), recursive=True))
    for p in files23:
        f = read_odim_composite(p, "DBZH").to_field(0.0)
        echo = np.isfinite(f) & (f > 0)
        lab, _ = ndimage.label(echo, structure=EIGHT)
        s_ = echo & (np.bincount(lab.ravel())[lab] <= SMALL_PX)
        small_n = s_.astype(np.int32) if small_n is None else small_n + s_
        echo_n = echo.astype(np.int32) if echo_n is None else echo_n + echo
    np.savez_compressed(SPECKLE, small=small_n, echo=echo_n, n_files=len(files23))
sp = np.load(SPECKLE)
iso = sp["small"] / int(sp["n_files"])
print(f"isolated echo pixel-times: {sp['small'].sum() / sp['echo'].sum():.2%} of the echo pixel-times")
for lo, hi in ((0, 20), (20, 60), (60, 120), (120, 1e9)):
    m = (dist_km > lo) & (dist_km <= hi)
    print(f"  {lo:>4.0f}-{hi:<5.0f} km from a radar: {sp['small'][m].sum() / sp['echo'][m].sum():.2%}")

az_all, ring_all = {}, {}
for n, (r, c) in radars.items():
    if 0 <= r < n_years.shape[0] and 0 <= c < n_years.shape[1]:
        d = np.hypot(yy - r, xx - c) * 2.0
        az_all[n] = (np.degrees(np.arctan2(xx - c, -(yy - r))) % 360).astype(int)
        ring_all[n] = (d >= 20) & (d <= 80)


def profile(field, n):
    return np.array([np.nanmedian(field[ring_all[n] & (az_all[n] == a)]) if (ring_all[n] & (az_all[n] == a)).any() else np.nan
                     for a in range(360)])


last = [m for m in months if "2018-09" <= m <= "2024-06"]
F = sum(monthly_catalogue(m)["freq"].sum(0).astype(float) for m in last)
V = sum(monthly_catalogue(m)["valid"].astype(float) for m in last)
echo_last = F / np.maximum(V, 1)
for n in sorted(az_all):
    p_ = profile(echo_last, n); ref_ = np.nanmedian(p_)
    print(f"{n}: azimuths below 90% of the median {int(np.sum(p_ < 0.9 * ref_))}, above 110% {int(np.sum(p_ > 1.1 * ref_))}")

edges = {}
for k in ("acc_before", "acc_afterGab", "acc_after"):
    tot_ = 0
    for n in az_all:
        p_ = profile(fv[k], n)
        nxt = np.r_[p_[1:], p_[0]]
        tot_ += int(np.nansum(np.abs(nxt - p_) / np.minimum(p_, nxt) > 0.10))
    edges[k] = tot_
print("spoke edges (> 10% between adjacent azimuths), 2023:", edges)

rows = []
for m in months:
    if m < "2013-04":
        continue
    m_ = monthly_catalogue(m)
    e_ = m_["freq"].sum(0) / np.maximum(m_["valid"], 1)
    rows.append((m, int((e_ > 0.6).sum()), np.mean(dist_km[e_ > 0.6] <= 20) if (e_ > 0.6).any() else np.nan, int((e_ > 0.9).sum())))
hits = pd.DataFrame(rows, columns=["month", "px_60", "near_share", "px_90"])
hits["period"] = pd.cut(pd.to_datetime(hits.month + "-01"), [pd.Timestamp("2013-03-01"), pd.Timestamp("2015-11-30"), pd.Timestamp("2018-07-31"), pd.Timestamp("2024-12-31")],
                        labels=["4/2013-11/2015", "12/2015-7/2018", "8/2018-6/2024"])
print(hits.groupby("period", observed=True).agg(months=("month", "size"), px_60_median=("px_60", "median"), px_60_max=("px_60", "max"),
                                                near_share=("near_share", "median"), px_90_median=("px_90", "median")))

fig, ax = plt.subplots(1, 2, figsize=(13, 5.6), layout="constrained")
im = ax[0].imshow(np.where(iso > 0, iso, np.nan), cmap="magma_r", norm=LogNorm(vmin=1e-4, vmax=3e-2)); fig.colorbar(im, ax=ax[0], shrink=0.8)
ax[0].set_title("ODYSSEY 2023: frequency of isolated echoes (<= 11 px)")
im = ax[1].imshow(np.where(sp["echo"] > 0, sp["small"] / np.maximum(sp["echo"], 1), np.nan), cmap="viridis", vmax=0.2); fig.colorbar(im, ax=ax[1], shrink=0.8)
ax[1].set_title("share of the pixel's echoes that are isolated")
for a_ in ax:
    for r, c in radars.values():
        a_.plot(c, r, "o", mfc="none", mec="red", ms=8, mew=1.2)
    a_.set_xlim(0, iso.shape[1]); a_.set_ylim(iso.shape[0], 0); a_.set_xticks([]); a_.set_yticks([])
plt.show()

## What is seen

Run on 9/11/2026.

- **Summer convective days:** the filter removes 4.9% of the convective rain at 2 km (4.0–6.3% per day) and
  2.3% at 1 km (1.3–3.0%); 0.9% and 0.6% of the stratiform rain.
- **January days with stationary features:** it removes 11.2% of the convective rain at 2 km (9.2–16.4% per
  day) and 7.1% at 1 km (6.1–10.3%): more than twice the summer share at 2 km, three times at 1 km. These
  "convective" pixels are mostly the stationary features, but not only them, so this is an upper bound of
  the gain on the features, not a measure of it.
- **Where the removed convective rain lies:** 92–93% inside echo regions, 5–7% on edges, 1–3% in small
  echoes. The filter acts mostly through its gradient test, on pixels much stronger than their neighbours,
  not by removing small cells.
- **Catalogue, ODYSSEY 2023 and CIRRUS 2025** (section 4):
  - The annual sums (median 2,831 and 2,204 mm) are far above the rain that falls in the region, because
    a column-maximum reflectivity converted with Marshall–Palmer is not a rain estimate; they serve only as
    a statistic for `histo_cut`.
  - With 5% tails, `histo_cut` flags 8,148 ODYSSEY pixels (3.4% of the crop) and 31,851 CIRRUS pixels
    (3.3%) as static clutter, and no shading. Most flagged areas are discs centred on the radars, where the
    annual sum is highest, plus the patches on and off the Dutch coast; the discs may be the effect of
    distance from the radar on a maximum composite rather than clutter (not checked).
  - The frequency of ≥ 55 dBZ shows the lines over the North Sea in both products, and radial lines in the
    south-east of ODYSSEY.
- **Catalogue, ODYSSEY 2013–2024** (sections 4–5, run on 10/11/2026):
  - `histo_cut` flags 6,042–12,172 pixels a year (8,131 in 2013, 6,042 in the half year 2024) and never any
    shading.
  - **The mask is not stable.** Only 631 pixels are flagged in all 12 years and 1,949 in 10 or more;
    15,748 are flagged in only 1 to 3 years. Of each year's mask, 59–83% was already flagged the year before.
  - The stable pixels are the centres of the discs and the fixed points. The outer rings of the discs change
    from year to year, as rain would.
  - **The discs sit on the radars:** of the 44 discs flagged in at least 6 years, 26 have their centre within
    10 km of a radar; 75% of the pixels flagged in at least 10 years lie within 20 km of a radar, against
    3.7% of the crop. They are the near-radar excess of a maximum composite, not clutter, so the static mask
    of the first version is to be based on the frequency maps and on the points above 70 dBZ (review of #74).
- **Radars over time** (section 7):
  - Two steps shared by almost all radars: the median near-radar echo frequency falls from 0.44 (November
    2015) to 0.22 (December 2015), and from 0.28 (July 2018) to 0.10 (August 2018), while the far one does
    not. December 2015 is the change of OPERA's central processing (satellite filter, beam-blockage
    correction; Saltikoff et al. 2019); August 2018 is consistent with the 2018 satellite filter of ODYSSEY
    (Radojevic 2024, Table 3), which gives the year, not the month.
  - Until 2015 the near-radar echo frequency reaches about 0.5 in summer, against 0.14–0.18 far away: echoes
    without rain near the radars, which the satellite filter removes.
  - Since 2019 the near/far ratio is about 1.2 in summer and winter, so the discs belong mostly to 2013–2018.
- **Isolated echoes, blocked sectors, spokes** (section 8):
  - 1.2% of the echo pixel-times of 2023 are isolated, 2.0% within 20 km of a radar and 0.9% beyond 120 km;
    they draw rings and radial lines around the radars and the edge of a coverage in the north-west.
  - No blocked sector in the azimuth profiles of 2018–2024: the maximum composite fills them from the
    neighbouring radars, and OPERA corrects blockage since late 2015.
  - Spoke edges on the 2023 annual sum: 1,219 before, 1,061 after Gabella, 993 after the first version; one
    year of rain also makes such edges, so this is an upper bound of the residual.
  - OPERA's monthly hit-count rule (60%) flags a median of 2,040 pixels a month before December 2015, 16 from
    December 2015 to July 2018 and 0 since August 2018.
- **First version on ODYSSEY 2023** (section 6, mask of 2022):
  - Rain removed, all pixels: 0.16% by > 70 dBZ, 5.12% by Gabella, 3.87% by the static mask, 9.15% in
    total. Convective pixels: 0.25%, 7.36%, 5.25%, 12.86%. Stratiform: 0.00%, 1.37%, 1.56%, 2.93%.
  - Over Belgium, 12.0% of the annual sum is removed (3,765 → 3,312 mm, index).
  - Echoes kept: 99.8% of those above 70 dBZ are removed (the 83 left are exactly 70.0 dBZ); 63% of those of
    55–70 dBZ are kept, 85% of 40–55, 94% of 20–40, 93% of 0–20.
  - Pixels with ≥ 55 dBZ in more than 0.1% of the times: 1,984 → 209; more than 1%: 166 → 0. The North Sea
    lines and the fixed points go; part of the radial lines in the south-east stays (interference residual).
  - **The mask of 2022 overshoots on 2023:** it leaves dark rings and holes around several radars in the
    annual sum after cleaning, and blank patches in the ≥ 55 dBZ frequency, where real strong echoes of 2023
    were lowered with the clutter. This follows from the instability of section 5.
  - Gabella removes 7.36% of the convective rain over the year, more than on the five summer days of section
    1 (4.9% at 2 km), because its share includes the fixed points that Steiner classes as convective.